In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel

In [ ]:
BATCH_SIZE = 16
LEARNING_RATE = 3e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("dair-ai/emotion", "split")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3


In [4]:
class MultiClassClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  # Labels should be integers: 0, 1, 2, ..., num_classes-1
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)  # Changed to scalar tensor of type long
        }

In [5]:
class BertForMultiClassClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiClassClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output size is num_classes
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits, no sigmoid

In [6]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [7]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size,)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # CrossEntropyLoss expects logits and long labels
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  # Get class indices
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds)
        train_true = np.array(train_true)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [8]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].item()
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = torch.argmax(output, dim=1).item()  # Scalar integer
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [9]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values  # Must be integers: 0, 1, 2, ..., num_classes-1

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

train_dataset = MultiClassClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiClassClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiClassClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]

results = []

# Get number of classes from the training data
num_classes = train_df['label'].nunique()

# Loop through seeds
for seed in seeds:
    torch.manual_seed(seed)
    model = BertForMultiClassClassification(num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    save_path = f'results/bert_multiclass3_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval

    # Load best model
    model.load_state_dict(torch.load(save_path))

    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_test_time = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    # Store individual results for this seed
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_test_time
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/500 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/4 - Train Loss: 0.6131, Acc: 0.7836, F1: [0.81498456 0.8346484  0.62030598 0.75       0.71650212 0.58114035]
Epoch 1/4 - Val Loss: 0.2019, Acc: 0.9210, F1: [0.94756554 0.94185212 0.86956522 0.92280072 0.86580087 0.82894737]
Model saved!


Epoch 2/4 - Train Loss: 0.1524, Acc: 0.9389, F1: [0.97095702 0.95551185 0.86252354 0.93815149 0.90744571 0.80902778]
Epoch 2/4 - Val Loss: 0.1483, Acc: 0.9330, F1: [0.96065874 0.95371669 0.86068111 0.93169877 0.88317757 0.8427673 ]
Model saved!


Epoch 3/4 - Train Loss: 0.1083, Acc: 0.9499, F1: [0.97932069 0.96399514 0.8739114  0.95317571 0.91975309 0.84173298]
Epoch 3/4 - Val Loss: 0.1645, Acc: 0.9300, F1: [0.95551601 0.95021337 0.85875706 0.93358634 0.88679245 0.83636364]


Epoch 4/4 - Train Loss: 0.0897, Acc: 0.9571, F1: [0.98319957 0.96754279 0.88872067 0.96521336 0.93188854 0.85714286]
Epoch 4/4 - Val Loss: 0.1631, Acc: 0.9330, F1: [0.9602122  0.9550173  0.85173502 0.92571429 0.88584475 0.84722222]


...

Epoch 2/4 - Train Loss: 0.1539, Acc: 0.9387, F1: [0.97096325 0.95424222 0.85810558 0.93777469 0.90993309 0.82167832]
Epoch 2/4 - Val Loss: 0.1597, Acc: 0.9295, F1: [0.95348837 0.94903087 0.86327078 0.93005671 0.8852459  0.8625    ]
Model saved!


Epoch 3/4 - Train Loss: 0.1073, Acc: 0.9497, F1: [0.97985426 0.96531089 0.8831758  0.95074349 0.91413882 0.82716049]
Epoch 3/4 - Val Loss: 0.1361, Acc: 0.9350, F1: [0.96489649 0.95298246 0.85798817 0.93862816 0.88619855 0.8427673 ]
Model saved!


Epoch 4/4 - Train Loss: 0.0895, Acc: 0.9585, F1: [0.98435491 0.97005876 0.89381207 0.96450939 0.93254681 0.85075961]
Epoch 4/4 - Val Loss: 0.1482, Acc: 0.9330, F1: [0.96087352 0.95299145 0.87078652 0.93214286 0.87529412 0.85897436]


C:\Users\Rafael\AppData\Local\Temp\ipykernel_5748\890430430.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafael

Test Time: 21.82 seconds
Test Metrics:
Accuracy: 0.929
F1s: [0.96763203 0.94721826 0.81578947 0.93235832 0.8929385  0.76119403]
Precisions: [0.95784148 0.93917963 0.85517241 0.9375     0.91162791 0.75      ]
Recalls: [0.97762478 0.95539568 0.77987421 0.92727273 0.875      0.77272727]


In [10]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multiclass3.csv', index=False)

In [11]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,32,0.00002,0.938875,"[0.9710610932475884, 0.9597365945437442, 0.847...","[0.9708529789969995, 0.9513241327862738, 0.878...","[0.9709570249705284, 0.9555118478973494, 0.862...",1289.750000,3806.450195,633.051235,...,"[0.9668508287292817, 0.9418282548476454, 0.958...","[0.9545454545454546, 0.9659090909090909, 0.780...","[0.9606587374199451, 0.9537166900420757, 0.860...",0.9300,"[0.9787610619469026, 0.9273972602739726, 0.948...","[0.9518072289156626, 0.9741007194244604, 0.698...","[0.9650959860383944, 0.9501754385964912, 0.804...",1225.792969,1761.220703,22.589563
1,3,32,0.00002,0.952000,"[0.9803040034253907, 0.9669979373710857, 0.870...","[0.9813544792113159, 0.9617679970160388, 0.888...","[0.9808289600514084, 0.9643758765778401, 0.879...",1308.832031,3827.325195,608.362111,...,"[0.9830508474576272, 0.9705014749262537, 0.860...","[0.9490909090909091, 0.9346590909090909, 0.898...","[0.96577243293247, 0.9522431259044862, 0.87912...",0.9265,"[0.987410071942446, 0.9699248120300752, 0.7932...","[0.9449225473321858, 0.9280575539568345, 0.893...","[0.9656992084432717, 0.9485294117647058, 0.840...",1228.312500,1781.720703,22.274340
2,5,32,0.00002,0.949688,"[0.979854264894985, 0.967935495968498, 0.87099...","[0.979854264894985, 0.9627004848936964, 0.8957...","[0.979854264894985, 0.9653108929406264, 0.8831...",1309.191406,3822.075195,608.391046,...,"[0.9554367201426025, 0.941747572815534, 0.9062...","[0.9745454545454545, 0.9644886363636364, 0.814...","[0.9648964896489649, 0.9529824561403509, 0.857...",0.9290,"[0.9578414839797639, 0.9391796322489392, 0.855...","[0.9776247848537005, 0.9553956834532374, 0.779...","[0.9676320272572402, 0.9472182596291013, 0.815...",536.597656,1778.345703,22.335803
